**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/mah-2023"], check=True)
    os.chdir("/content/encore/entries/mah-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Mah et al. (2023) from DANDI:001169: fits from more starts

Paper: Mah, Schiereck, Bossio and Constantinople (2023), "Distinct value computations support rapid sequential decisions", Nature Communications 14, 7573, doi:10.1038/s41467-023-43250-x.

Dataset: DANDI:001169, version 0.261120.1450 (doi:10.48324/dandi.001169/0.261120.1450), as in the other two notebooks. Claims file: `claims/mah-2023.yaml`.

`models_mah_2023.ipynb` fits the wait time models from 2 random starts of L-BFGS-B; the paper used fmincon from 100. The verifier asked whether the two outcomes that sit near a threshold or turn on the fitted parameters depend on this: suppfig8-belief-state-not-distinguished (lowest p = 0.062, under seed +3000) and fig3f-inferential-model-not-sensitive-to-previous-offer. This notebook refits from 6 starts the inferential and belief state models under seed +3000, the seed nearest the threshold for suppfig8, and the inferential model under the claim's seed, the fits behind fig3f. Everything else is as in `models_mah_2023.ipynb`; the folds are drawn first from the same generator, so they are those of the 2-start fits there. 6 starts is what fits within the time of a run of the entry; 100 would take about 17 times as long. No outcome is decided here; the outcomes stay those of `models_mah_2023.ipynb`.

| Claim | Checked here |
|---|---|
| suppfig8-belief-state-not-distinguished | with 6 starts, seed +3000 |
| fig3f-inferential-model-not-sensitive-to-previous-offer | with 6 starts, claim's seed |

## Data

The same loading as the other notebooks.

In [ ]:
import os
for v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[v] = "1"  # many small fits in parallel processes: one linear-algebra thread each
import zlib, time
import numpy as np
import pandas as pd
from scipy import stats
import mah2023 as M

def rng_for(claim, k=0, offset=0):
    """The claim's generator (protocol: seed = crc32 of '<paper key>/<claim id>'), plus `offset`
    for the seed-sensitivity runs (+1000 ... +4000) and k for a second generator."""
    return np.random.default_rng(zlib.crc32(f"mah-2023/{claim}".encode()) + k + offset)

T0 = time.time()

In [ ]:
assets = M.list_assets()
sessions, trials = M.load_all(assets)
t = M.add_measures(trials)
rats = np.sort(t["subject"].unique())

In [ ]:
print("Dandiset", M.DANDISET, "version", M.VERSION, "| assets:", len(assets))
print(f"loaded {len(sessions)} sessions, {len(t)} trials of {len(rats)} rats in {time.time()-T0:.0f} s")
print("trials with a wait time (model trials):", int(t.wait_time.notna().sum()), "| opt-out catch trials with a wait time:", int(t.wt.notna().sum()))

## Fits from 6 starts

In [ ]:
MODEL_RATS = list(rats)
SEED = zlib.crc32(b"mah-2023/fig3j-inferential-beats-retrospective-for-wait-times")
NS = 6
FITS = {}
for off, models in [(3000, ["inferential", "belief"]), (0, ["inferential"])]:
    TS = time.time()
    FITS[off] = M.fit_all(t, MODEL_RATS, models, SEED + off, nstarts=NS)
    print(f"seed +{off}, {NS} starts, {' and '.join(models)}: fitted {len(FITS[off])} rats in {time.time()-TS:.0f} s", flush=True)

def metric_table(F):
    rows = []
    for r, d in F.items():
        for m, v in d.items():
            rows.append(dict(rat=r, model=m, nll=v["nll_all"], k=v["k"], n=v["n"], cv=v["cv_nll"], cvbest=v["best_fold_test_nll"]))
    df = pd.DataFrame(rows)
    df["BIC"] = np.log(df.n) * df.k + 2 * df.nll
    df["AIC"] = 2 * df.k + 2 * df.nll
    return df.pivot(index="rat", columns="model")

MT = {k: metric_table(F) for k, F in FITS.items()}
P = pd.DataFrame({r: FITS[0][r]["inferential"]["params"] for r in MODEL_RATS}, index=["k_mixed", "k_high", "k_low", "D"]).T
print(f"inferential parameters (claim's seed, {NS} starts), median:", P.median().round(3).to_dict(), "| at a bound:",
      {c: int(((P[c] <= 0.0101) | (P[c] >= 4.99)).sum()) for c in P})
for off, mt in MT.items():
    print(f"seed +{off}, {NS} starts: summed nLL over rats on all data, " + ", ".join(
          f"{m} {mt['nll'][m].sum():.1f}" for m in mt['nll'].columns) + "; summed test nLL, " + ", ".join(
          f"{m} {mt['cv'][m].sum():.1f}" for m in mt['cv'].columns))

## suppfig8-belief-state-not-distinguished

> Model comparison does not distinguish the inferential model from the belief state model for wait times.

The comparison of `models_mah_2023.ipynb` (signed-rank test across rats of the inferential minus belief state difference in BIC, AIC, summed test nLL and, as a variant, best-fold test nLL), from 6 starts under seed +3000; the 2-start results are printed in `models_mah_2023.ipynb`.

In [ ]:
for off, mt in [(3000, MT[3000])]:
    out = []
    for met in ["BIC", "AIC", "cv", "cvbest"]:
        d = mt[met]["inferential"] - mt[met]["belief"]
        out.append(f"{met}: mean diff {d.mean():+.2f}, inferential better {int((d < 0).sum())}/{len(d)}, p = {stats.wilcoxon(d).pvalue:.3g}")
    print(f"seed +{off}, {NS} starts: " + " | ".join(out))

## fig3f-inferential-model-not-sensitive-to-previous-offer

> Wait times predicted by the inferential model fitted to each rat are, like the rats' wait times, not sensitive to the previous offer.

As in `models_mah_2023.ipynb`: one log-normal draw (variance 8) per trial from each rat's inferential fit under the claim's seed, put in place of the rat's wait time and tested as in fig2j (two-tailed rank-sum on the per-rat means after offers below and above 20 uL), under the five draw seeds; and the model's mean wait time without noise. From 6 starts; the 2-start results are printed in `models_mah_2023.ipynb`.

In [ ]:
RMS = {r: M.RatModel(g) for r, g in t.groupby("subject")}

def model_wait_times(F, rng, noise=True):
    out = pd.Series(np.nan, index=t.index)
    for r in MODEL_RATS:
        rm = RMS[r]
        w = rm.wt_opt("inferential", F[r]["inferential"]["params"], only_fit=False)
        if noise:
            s2 = np.log(M.NOISE_WT / w ** 2 + 1)
            w = np.exp(rng.normal(np.log(w) - s2 / 2, np.sqrt(s2)))
        out.loc[rm.index[rm.fit]] = w[rm.fit]
    return out

def fig2j_test(wcol):
    sel = (t.vol == 20) & (t.block == 1) & wcol.notna() & t.prev_vol.notna()
    w = t.loc[sel, ["subject", "prev_vol"]].assign(v=wcol[sel])
    w["zz"] = w.groupby("subject").v.transform(lambda x: (x - x.mean()) / x.std())
    m = pd.DataFrame({"below": w[w.prev_vol < 20].groupby("subject").zz.mean(), "above": w[w.prev_vol > 20].groupby("subject").zz.mean()})
    return M.ranksum_p(m.below, m.above), (m.below - m.above).mean()

for ns in [NS]:
    F = FITS[0]
    res = [fig2j_test(model_wait_times(F, rng_for("fig3f-inferential-model-not-sensitive-to-previous-offer", offset=off)))
           for off in [0, 1000, 2000, 3000, 4000]]
    p0, s0 = fig2j_test(model_wait_times(F, None, noise=False))
    print(f"{ns} starts: draw seeds +0 to +4000, sensitivity " + ", ".join(f"{s:+.4f}" for _, s in res)
          + "; rank-sum p " + ", ".join(f"{p:.3g}" for p, _ in res) + f" | without noise {s0:+.4f}, p = {p0:.3g}")
print(f"total {time.time()-T0:.0f} s")

**Result: no outcome changes.** Under seed +3000, fits from 6 starts give BIC and AIC p = 0.0507 for suppfig8 (inferential model better in 127 of 291 rats), against 0.0305 from 2 starts on this machine, and cross-validated nLL p = 0.561 (0.459 from 2 starts). The extra starts lower the summed nLL on all data little (inferential 3,450,200.6 against 3,450,295.8; belief 3,449,887.0 against 3,449,887.2) and the summed test nLL more for the inferential model (3,479,562.3 against 3,485,539.0) than for the belief state model (3,479,223.2 against 3,479,837.0). So suppfig8 stays near the threshold under this seed. For fig3f, the claim's seed fits from 6 starts give the model a sensitivity of +0.048 z (rank-sum p = 3.6e-05; the four other draws 0.328, 4.7e-07, 1.6e-05 and 4.2e-04) and +0.424 without noise (p = 9.2e-62), as from 2 starts, so the failure of fig3f does not come from too few starts. 6 starts are three times the 2 of `models_mah_2023.ipynb` and far from the paper's 100.